In [11]:
#!/usr/bin/env python3
import torch
from torchdiffeq import odeint

torch.set_default_dtype(torch.float64)
device = 'cpu'

# Toy field: v(x,t) = alpha * x  -> div = alpha * d
alpha = 0.3
d = 3

def f(t, state):
    x, logp = state
    v = alpha * x
    div = alpha * d * torch.ones(x.shape[0], dtype=x.dtype, device=x.device)
    dxdt  = v
    dlpdt = -div                    # <-- SAME for forward and reverse
    return dxdt, dlpdt

# Base distribution: standard normal
B = 1024
x0 = torch.randn(B, d, device=device)
logp0 = -0.5*(x0**2).sum(-1) - 0.5*d*torch.log(torch.tensor(2*torch.pi))             # log N(0,I)

# Time grids
t_fwd = torch.tensor([0., 1.], device=device)
t_rev = torch.flip(t_fwd, dims=[0])                                     # [1, 0]

# FORWARD: push base to t=1 and accumulate logp
(x1, lp1) = odeint(f, (x0, logp0), t_fwd, method='rk4',
                   options={'step_size': 1/200})
x1 = x1[-1]
lp1 = lp1[-1]
# Analytic forward solution for sanity:
# x1 = exp(alpha) * x0,   logp shift = - alpha * d
x1_true = torch.exp(torch.tensor(alpha)) * x0
lp1_true = logp0 + (-alpha*d)


In [14]:

print("forward: max|x1-x1_true| =", (x1-x1_true).abs().max().item())
print("forward: max|lp1-lp1_true| =", (lp1-lp1_true).abs().max().item())

# REVERSE (your pattern): flip the time grid; keep RHS signs unchanged
(x0_back, lp0_back) = odeint(f, (x1, lp1), t_rev, method='rk4',
                             options={'step_size': 1/200})
x0_back = x0_back[-1]
lp0_back = lp0_back[-1]
print("round-trip: max|x0_back-x0| =", (x0_back-x0).abs().max().item())
print("round-trip: max|lp0_back-logp0| =", (lp0_back-logp0).abs().max().item())

# Now compute estimate_logprob(x1) as base_logprob(z) + integral with flipped grid.
# Running the reverse solve gives us z and the accumulated scalar already in lp0_back.
# Since lp tracked 'forward-time' dlogp/dt=-div, integrating on [1,0] yields the correct
# +∫_0^1(-div)dt shift automatically. So:
base_logprob_of_z = -0.5*(x0_back**2).sum(-1) - 0.5*d*torch.log(torch.tensor(2*torch.pi))
est_logprob = base_logprob_of_z + (lp1 - base_logprob_of_z)  # equals lp1
print("estimate vs pushed: max|est - lp1| =", (est_logprob - lp1).abs().max().item())


forward: max|x1-x1_true| = 6.661338147750939e-14
forward: max|lp1-lp1_true| = 3.4638958368304884e-14
round-trip: max|x0_back-x0| = 2.6645352591003757e-15
round-trip: max|lp0_back-logp0| = 9.325873406851315e-15
estimate vs pushed: max|est - lp1| = 0.0
